# Project 4 Data Dictionary

Nepal Earthquake Damage Database

# Overview

This data dictionary describes the Nepal earthquake damage data used in
Project 4. The data is available in two formats:

1.  **`nepal.sqlite`** - A SQLite database file containing all four
    tables
2.  **Four CSV files** in the `data/` folder:
    -   `building_structure.csv` - Building characteristics (234,835
        rows)
    -   `building_damage.csv` - Damage assessments (234,835 rows)
    -   `household_demographics.csv` - Household information (249,932
        rows)
    -   `id_map.csv` - Bridge table linking households to buildings and
        districts (249,932 rows)

Both formats contain the same data and can be queried using DuckDB
(recommended) or SQLite.

**Source**: Data collected by the National Planning Commission of Nepal
following the 2015 Gorkha earthquake.

**Note**: This dataset uses a **simplified subset** with 4 districts
instead of the original 11 from the full Kaggle dataset.

---

## 1. District ID Mapping

The database contains data from **4 districts** with simplified numeric
IDs:

| Database ID | District Name | Building Count | Household Count | Key Characteristics |
|------------|--------------|--------------|---------------|------------------|
| **1** | Sindhupalchok | 35,285 | 36,112 | Chhetree dominant, significant Brahman-Hill and Rai populations |
| **2** | Okhaldhunga | 52,181 | 55,253 | Mixed Chhetree/Tamang/Newar/Magar population |
| **3** | Kavrepalanchok | **76,533** | 82,684 | **Largest district**; Tamang and Brahman-Hill dominant |
| **4** | Gorkha | 70,836 | 75,883 | Gurung dominant (unique to this district); Kumal population present |

## How to Identify Districts

Since the database stores only numeric IDs, you can identify districts
by querying demographic patterns:

**Example: Query caste distribution by district using DuckDB**

## Reference, not a lesson

P4-L6 is the **data dictionary** for the Nepal earthquake project. It is a reference document, not a teaching lesson: there are no Code Tasks, no modelling work, no Multiple Choice Questions. The intended use is: when you are working in L1–L5 and a column name confuses you, or you are unsure which table holds which information, or you want to verify the expected values of a categorical variable, you flip to this dictionary.

The document is organised in three sections: (1) the **district-ID mapping** that translates the numeric `district_id` field into the named districts (Sindhupalchok, Okhaldhunga, Kavrepalanchok, Gorkha) and tells you how to identify a district from its caste distribution when you only have the numeric ID; (2) the **table schemas** — the exact columns of `building_structure`, `building_damage`, `household_demographics`, and `id_map`, with each column's expected type and example values; (3) **join keys** — which columns connect the four tables so you can reconstruct the per-building view L1's `wrangle_nepal_data` function delivers.

In [ ]:
import duckdb

# Query caste distribution by district from CSV files
df = duckdb.sql("""
    SELECT i.district_id, h.caste_household, COUNT(*) as count
    FROM read_csv_auto('./data/household_demographics.csv') h
    JOIN read_csv_auto('./data/id_map.csv') i 
        ON h.household_id = i.household_id
    GROUP BY i.district_id, h.caste_household
    ORDER BY i.district_id, count DESC
""").fetchdf()

# Show top 5 castes per district
for district in [1, 2, 3, 4]:
    print(f"\nDistrict {district}:")
    print(df[df['district_id'] == district].head(5))

**Key Identifiers:**

-   **District 4 (Gorkha)**: Only district with Gurung as the #1 caste
-   **District 3 (Kavrepalanchok)**: Largest district (76,533
    buildings), Tamang dominant
-   **District 1 (Sindhupalchok)**: Significant Sherpa population
-   **District 2 (Okhaldhunga)**: Balanced mix of major castes

---

## 2. Database Tables

The data is organized into four related tables (or CSV files):

## 1. `building_structure`

Contains structural characteristics of buildings.

| Variable | Description | Type |
|-------------------------|--------------------------------|----------------|
| `building_id` | Unique identifier for each building | INTEGER |
| `count_floors_pre_eq` | Number of floors before earthquake | INTEGER |
| `count_floors_post_eq` | Number of floors after earthquake | INTEGER |
| `age_building` | Age of building in years | INTEGER |
| `plinth_area_sq_ft` | Plinth area in square feet | INTEGER |
| `height_ft_pre_eq` | Height in feet before earthquake | INTEGER |
| `height_ft_post_eq` | Height in feet after earthquake | INTEGER |
| `land_surface_condition` | Surface condition of land (Flat, Moderate slope, Steep slope) | TEXT |
| `foundation_type` | Type of foundation | TEXT |
| `roof_type` | Type of roof | TEXT |
| `ground_floor_type` | Ground floor construction type | TEXT |
| `other_floor_type` | Other floors construction type | TEXT |
| `position` | Building position (Attached, Not attached) | TEXT |
| `plan_configuration` | Building plan shape | TEXT |
| `condition_post_eq` | Post-earthquake condition | TEXT |
| `superstructure` | Superstructure materials | TEXT |

**Example values:**

-   `foundation_type`: “Mud mortar-Stone/Brick”, “Cement-Stone/Brick”,
    “Other”, “RC”
-   `roof_type`: “Bamboo/Timber-Light roof”, “Bamboo/Timber-Heavy roof”,
    “RCC/RB/RBC”
-   `damage_grade`: “Grade 1”, “Grade 2”, “Grade 3”, “Grade 4”, “Grade
    5”

## 2. `building_damage`

Contains detailed damage assessment data.

| Variable | Description | Type |
|-------------------------|--------------------------------|----------------|
| `building_id` | Unique identifier for each building | INTEGER |
| `damage_grade` | Overall damage grade (Grade 1-5) | TEXT |
| `damage_overall_collapse` | Collapse damage assessment | TEXT |
| `damage_overall_leaning` | Leaning damage assessment | TEXT |
| `damage_overall_adjacent_building_risk` | Adjacent building risk | TEXT |
| `damage_foundation_severe` | Severe foundation damage proportion | TEXT |
| `damage_foundation_moderate` | Moderate foundation damage proportion | TEXT |
| `damage_foundation_insignificant` | Insignificant foundation damage proportion | TEXT |
| `damage_roof_severe` | Severe roof damage proportion | TEXT |
| `damage_roof_moderate` | Moderate roof damage proportion | TEXT |
| `damage_roof_insignificant` | Insignificant roof damage proportion | TEXT |
| `damage_corner_separation_severe` | Severe corner separation damage | TEXT |
| `damage_corner_separation_moderate` | Moderate corner separation damage | TEXT |
| `damage_corner_separation_insignificant` | Insignificant corner separation damage | TEXT |
| `damage_diagonal_cracking_severe` | Severe diagonal cracking damage | TEXT |
| `damage_diagonal_cracking_moderate` | Moderate diagonal cracking damage | TEXT |
| `damage_diagonal_cracking_insignificant` | Insignificant diagonal cracking damage | TEXT |
| `damage_in_plane_failure_severe` | Severe in-plane failure damage | TEXT |
| `damage_in_plane_failure_moderate` | Moderate in-plane failure damage | TEXT |
| `damage_in_plane_failure_insignificant` | Insignificant in-plane failure damage | TEXT |
| `damage_out_of_plane_failure_severe` | Severe out-of-plane failure damage | TEXT |
| `damage_out_of_plane_failure_moderate` | Moderate out-of-plane failure damage | TEXT |
| `damage_out_of_plane_failure_insignificant` | Insignificant out-of-plane failure damage | TEXT |
| `damage_out_of_plane_failure_walls_ncfr_severe` | Severe out-of-plane failure of walls not carrying floor/roof | TEXT |
| `damage_out_of_plane_failure_walls_ncfr_moderate` | Moderate out-of-plane failure of walls not carrying floor/roof | TEXT |
| `damage_out_of_plane_failure_walls_ncfr_insignificant` | Insignificant out-of-plane failure of walls not carrying floor/roof | TEXT |
| `damage_gable_failure_severe` | Severe gable failure damage | TEXT |
| `damage_gable_failure_moderate` | Moderate gable failure damage | TEXT |
| `damage_gable_failure_insignificant` | Insignificant gable failure damage | TEXT |
| `damage_delamination_failure_severe` | Severe delamination failure damage | TEXT |
| `damage_delamination_failure_moderate` | Moderate delamination failure damage | TEXT |
| `damage_delamination_failure_insignificant` | Insignificant delamination failure damage | TEXT |
| `damage_column_failure_severe` | Severe column failure damage | TEXT |
| `damage_column_failure_moderate` | Moderate column failure damage | TEXT |
| `damage_column_failure_insignificant` | Insignificant column failure damage | TEXT |
| `damage_beam_failure_severe` | Severe beam failure damage | TEXT |
| `damage_beam_failure_moderate` | Moderate beam failure damage | TEXT |
| `damage_beam_failure_insignificant` | Insignificant beam failure damage | TEXT |
| `damage_infill_partition_failure_severe` | Severe infill/partition failure damage | TEXT |
| `damage_infill_partition_failure_moderate` | Moderate infill/partition failure damage | TEXT |
| `damage_infill_partition_failure_insignificant` | Insignificant infill/partition failure damage | TEXT |
| `damage_staircase_severe` | Severe staircase damage | TEXT |
| `damage_staircase_moderate` | Moderate staircase damage | TEXT |
| `damage_staircase_insignificant` | Insignificant staircase damage | TEXT |
| `damage_parapet_severe` | Severe parapet damage | TEXT |
| `damage_parapet_moderate` | Moderate parapet damage | TEXT |
| `damage_parapet_insignificant` | Insignificant parapet damage | TEXT |
| `damage_cladding_glazing_severe` | Severe cladding/glazing damage | TEXT |
| `damage_cladding_glazing_moderate` | Moderate cladding/glazing damage | TEXT |
| `damage_cladding_glazing_insignificant` | Insignificant cladding/glazing damage | TEXT |
| `area_assesed` | Areas assessed (Both, Exterior, Interior) | TEXT |
| `technical_solution_proposed` | Proposed technical solution | TEXT |
| `has_repair_started` | Whether repair work has started | REAL |
| `has_damage_foundation` | Flag for foundation damage | REAL |
| `has_damage_roof` | Flag for roof damage | REAL |
| `has_damage_corner_separation` | Flag for corner separation damage | REAL |
| `has_damage_diagonal_cracking` | Flag for diagonal cracking damage | REAL |
| `has_damage_in_plane_failure` | Flag for in-plane failure damage | REAL |
| `has_damage_out_of_plane_failure` | Flag for out-of-plane failure damage | REAL |
| `has_damage_out_of_plane_walls_ncfr_failure` | Flag for out-of-plane failure of walls not carrying floor/roof | REAL |
| `has_damage_gable_failure` | Flag for gable failure damage | REAL |
| `has_damage_delamination_failure` | Flag for delamination failure damage | REAL |
| `has_damage_column_failure` | Flag for column failure damage | REAL |
| `has_damage_beam_failure` | Flag for beam failure damage | REAL |
| `has_damage_infill_partition_failure` | Flag for infill/partition failure damage | REAL |
| `has_damage_staircase` | Flag for staircase damage | REAL |
| `has_damage_parapet` | Flag for parapet damage | REAL |
| `has_damage_cladding_glazing` | Flag for cladding/glazing damage | REAL |
| `has_geotechnical_risk` | Flag for geotechnical risk | REAL |
| `has_geotechnical_risk_land_settlement` | Flag for land settlement risk | INTEGER |
| `has_geotechnical_risk_fault_crack` | Flag for fault crack risk | INTEGER |
| `has_geotechnical_risk_liquefaction` | Flag for liquefaction risk | INTEGER |
| `has_geotechnical_risk_landslide` | Flag for landslide risk | INTEGER |
| `has_geotechnical_risk_rock_fall` | Flag for rock fall risk | INTEGER |
| `has_geotechnical_risk_flood` | Flag for flood risk | INTEGER |
| `has_geotechnical_risk_other` | Flag for other geotechnical risks | INTEGER |

**Damage Grade Definitions:**

-   **Grade 1**: No damage
-   **Grade 2**: Minor damage
-   **Grade 3**: Moderate damage
-   **Grade 4**: Severe damage
-   **Grade 5**: Collapse/destruction

**For binary classification:**

-   `severe_damage = 1`: Grade 4 or Grade 5
-   `severe_damage = 0`: Grade 1, 2, or 3

## 3. `household_demographics`

Contains demographic information about households.

| Variable | Description | Type |
|-------------------------|--------------------------------|----------------|
| `household_id` | Unique identifier for each household | INTEGER |
| `gender_household_head` | Gender of household head (Male, Female) | TEXT |
| `age_household_head` | Age of household head | REAL |
| `caste_household` | Caste/ethnicity of household | TEXT |
| `education_level_household_head` | Education level of household head | TEXT |
| `income_level_household` | Income level | TEXT |
| `size_household` | Number of people in household | REAL |
| `is_bank_account_present_in_household` | Whether household has bank account (1=Yes, 0=No) | REAL |

**What is Caste?**

In Nepal, “caste” refers to **ethnic and social groups** that
traditionally have distinct cultural practices, occupations, and
geographic concentrations. Unlike some other countries, Nepal’s caste
system is complex and includes both traditional Hindu caste categories
and ethnic/tribal groups (Janajati).

**Common Caste/Ethnic Groups in This Dataset:**

-   **Brahman-Hill** - Traditional priestly/scholarly caste
-   **Chhetree** - Traditional warrior/administrator caste (largest
    group overall)
-   **Tamang** - Major ethnic group, **dominant in Kavrepalanchok
    (district 3)**
-   **Magar** - Ethnic group found across multiple districts
-   **Newar** - Indigenous group of the Kathmandu Valley
-   **Gurung** - Ethnic group, **dominant in Gorkha (district 4)** -
    *only district with Gurung as #1*
-   **Rai** - Ethnic group primarily in Eastern Nepal
-   **Sherpa** - Ethnic group, **concentrated in Sindhupalchok (district
    1)**
-   **Kumal** - Ethnic group, **found in Gorkha (district 4)**
-   **Kami, Sarki, Damai/Dholi** - Traditional occupational castes

**Why This Matters for District Identification:**

Each district has a unique caste distribution “fingerprint”:

-   **District 4 (Gorkha)**: Gurung is the #1 caste (unique identifier!)
-   **District 3 (Kavrepalanchok)**: Tamang is the #1 caste
-   **District 1 (Sindhupalchok)**: Significant Sherpa population
-   **District 2 (Okhaldhunga)**: More balanced distribution

## 4. `id_map`

Bridge table that links households to buildings and districts.

| Variable | Description | Type |
|-------------------------|--------------------------------|----------------|
| `household_id` | Unique identifier for household | INTEGER |
| `building_id` | Unique identifier for building | INTEGER |
| `vdcmun_id` | Village Development Committee/Municipality ID (1-40) | INTEGER |
| `district_id` | District ID (1-4, see mapping table above) | INTEGER |

**VDC/Municipality Ranges:**

-   District 1 (Sindhupalchok): vdcmun_id 1-8
-   District 2 (Okhaldhunga): vdcmun_id 9-16
-   District 3 (Kavrepalanchok): vdcmun_id 17-28
-   District 4 (Gorkha): vdcmun_id 30-40

---

## 3. Quick Reference

## District Summary Statistics

| Metric | District 1 (Sindhupalchok) | District 2 (Okhaldhunga) | District 3 (Kavrepalanchok) | District 4 (Gorkha) |
|------|------------------|-----------------|------------------|--------------|
| **Buildings** | 35,285 | 52,181 | 76,533 | 70,836 |
| **Households** | 36,112 | 55,253 | 82,684 | 75,883 |
| **VDCs/Municipalities** | 8 | 8 | 12 | 11 |
| **Dominant Caste** | Chhetree | Chhetree | Tamang | Gurung |

## Key DuckDB Queries

DuckDB can query data using **two approaches**:

### Approach A: Query SQLite Database

Use when you need to run multiple queries or want traditional SQL
syntax:

**List all tables:**

``` python
import duckdb
conn = duckdb.connect('./nepal.sqlite')
tables = conn.execute("""
    SELECT name FROM sqlite_master WHERE type='table'
""").fetchdf()
conn.close()
```

**Count buildings by district:**

``` python
import duckdb
conn = duckdb.connect('./nepal.sqlite')
district_counts = conn.execute("""
    SELECT district_id, COUNT(DISTINCT building_id) as building_count
    FROM id_map
    GROUP BY district_id
    ORDER BY district_id
""").fetchdf()
conn.close()
```

### Approach B: Query CSV Files Directly (Recommended)

Use for single queries - no connection management needed:

**Count buildings by district:**

``` python
import duckdb

district_counts = duckdb.sql("""
    SELECT district_id, COUNT(DISTINCT building_id) as building_count
    FROM './data/id_map.csv'
    GROUP BY district_id
    ORDER BY district_id
""").fetchdf()
```

**Get Gorkha data (district_id = 4):**

``` python
import duckdb

gorkha_data = duckdb.sql("""
    SELECT s.*, d.damage_grade
    FROM read_csv_auto('./data/building_structure.csv') s
    JOIN read_csv_auto('./data/building_damage.csv') d 
        ON s.building_id = d.building_id
    JOIN read_csv_auto('./data/id_map.csv') i 
        ON s.building_id = i.building_id
    WHERE i.district_id = 4
""").fetchdf()
```

**Get Kavrepalanchok data (district_id = 3):**

``` python
import duckdb

kavre_data = duckdb.sql("""
    SELECT s.*, d.damage_grade
    FROM read_csv_auto('./data/building_structure.csv') s
    JOIN read_csv_auto('./data/building_damage.csv') d 
        ON s.building_id = d.building_id
    JOIN read_csv_auto('./data/id_map.csv') i 
        ON s.building_id = i.building_id
    WHERE i.district_id = 3
""").fetchdf()
```

---

# Data Source

**Original Dataset**: [Kaggle - Predicting Building Damage Grade by
Earthquake](https://www.kaggle.com/code/gitanjali1425/predicting-building-damage-grade-by-earthquake)

**Collected by**: National Planning Commission of Nepal

**Coverage**: 11 districts affected by the 2015 Gorkha earthquake

**This Database**: Subset containing 4 representative districts with
simplified IDs

---

# Notes for Students

1.  **Always filter by district_id** when querying specific districts
2.  **Use id_map table** to join households with buildings and districts
3.  **District 4 = Gorkha** is used in Lessons 2-4
4.  **District 3 = Kavrepalanchok** is used in the Assignment (Lesson 5)
5.  **Create severe_damage target** using:
    `damage_grade.isin(['Grade 4', 'Grade 5'])`
6.  **Drop post-earthquake columns** (those with ‘post_eq’ in name) to
    avoid data leakage

## Summary

P4-L6 is the project-wide data dictionary. The four tables (`building_structure`, `building_damage`, `household_demographics`, `id_map`) plus the district-ID mapping (1 = Sindhupalchok, 2 = Okhaldhunga, 3 = Kavrepalanchok, 4 = Gorkha) are what every other lesson in P4 sits on top of. The `wrangle_nepal_data` function from L1 encapsulates the standard JOIN + filter + target-engineering pattern; this reference exists so you can audit each step or write a custom query when the wrangle function is not enough.

## Reflection Questions

Before moving on, consider these questions:

1.  Suppose you wanted to extend the analysis to a fifth district not in this dataset. Which of the four tables would you need new rows in, and what `district_id` would the new district get?
2.  The `damage_grade` column has values "Grade 1" through "Grade 5". The `severe_damage` target used throughout P4 collapses these into a binary 0/1. If you were running a separate analysis where the **graded** distinction mattered (e.g., insurance claims that pay out differently per grade), which of the two — `damage_grade` or `severe_damage` — would you use, and what additional pre-processing would the choice require?